# Motion Analysis Mid-Term Assignment

## Objective
This notebook focuses on performing motion analysis through change detection and blob matching on video sequences.

## Code Structure
The notebook processes videos stored in the `/video` folder and applies a sequence of image processing techniques to them.  

Each step of the pipeline produces an output video, which is saved in the `video/output` folder.  

In [ ]:
import cv2 
import numpy as np
import os

os.makedirs("video/output", exist_ok=True)

## Helper Function: Video Display

A helper function `show_video` is used to display the resulting video directly under each code block.

This allows for immediate visualization of the output at every stage of the processing pipeline.

If the visualization does not work correctly within the notebook, all output videos are still saved in the `/video/output` folder and can be viewed from there.

In [ ]:
from IPython.display import HTML, display

def show_videos(*items, min_width=200):
    def box(path, title):
        media = f'<video style="width:100%;" autoplay loop muted playsinline><source src="{path}" type="video/mp4"></video>'
        return f"""
        <div style="flex:1; min-width:{min_width}px; padding:10px; background:white; font-family:sans-serif; text-align:center; margin:5px; box-sizing:border-box;">
            <p style="margin:0 0 8px 0; font-size:14px; color:#333;">{title}</p>
            {media}
        </div>"""
    display(HTML(f'<div style="display:flex; flex-wrap:nowrap; width:100%; background:white;">{"".join(box(p, t) for p, t in items)}</div>'))

## Video Loading and Background Initialization

First, the input video is selected by specifying its path:
`video_path = "video/video.mp4"`  
(this path can be modified to process a different video).

The video is then read, and its main properties (such as frame size and frame rate) are extracted. These parameters are used to correctly generate the output videos.

As an initial processing step, the video frames are converted to grayscale.  
Then, each frame is computed as the average of the previous 10 frames from the original video.

The resulting video, representing this background initialization process, is saved as **"background_video_avg"**.

In [ ]:
Video_path = "video/video.mp4"
Avg_bg_video_path = "video/output/background_video_avg.mp4"

video = cv2.VideoCapture(Video_path)
fps = int(video.get(cv2.CAP_PROP_FPS))
w = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"avc1")
avg_video_out = cv2.VideoWriter(Avg_bg_video_path, fourcc, fps, (w, h), isColor=False)

frame_buffer = []
buffer_size = 10
frame_count = 0

while True:
    ret, frame = video.read()
    if not ret:
        break
    
    frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY).astype(np.float32)
    frame_buffer.append(frame_gray)
    
    if len(frame_buffer) > buffer_size:
        frame_buffer.pop(0)
    
    # Always write a frame for every input frame
    if len(frame_buffer) == 1:
        # For first frame, just use the frame itself
        bg_avg = frame_gray.astype(np.uint8)
    else:
        # Use average of available frames
        bg_avg = np.mean(frame_buffer, axis=0).astype(np.uint8)
    
    avg_video_out.write(bg_avg)
    
    frame_count += 1

video.release()
avg_video_out.release()

show_videos(
    (Video_path, "Original Video"),
    (Avg_bg_video_path, f"Average Background")
)

## Change Detection

In this step, the original video is compared with the previously computed average background.

The original video frames are converted to grayscale and, for each frame, a pixel-wise comparison with the background is performed.  
If the difference between a pixel and the corresponding background pixel exceeds a defined threshold, it is marked as white (foreground); otherwise, it is marked as black (background).

This produces a binary change detection video highlighting moving regions.

The resulting video is saved as **"video_change_avg"**

In [ ]:
threshold = 25
change_video_path = "video/output/video_change_avg.mp4"

video = cv2.VideoCapture(Video_path)
bg_video = cv2.VideoCapture(Avg_bg_video_path)

out = cv2.VideoWriter(change_video_path, fourcc, fps, (w, h))

while True:
    ret, frame = video.read()
    _, bg_frame = bg_video.read()
    if not ret:
        break    
    frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    bg_gray = cv2.cvtColor(bg_frame, cv2.COLOR_BGR2GRAY)
    
    diff = cv2.absdiff(frame_gray, bg_gray)
    _, mask = cv2.threshold(diff, threshold, 255, cv2.THRESH_BINARY)
    out.write(mask)

video.release()
bg_video.release()
out.release()

show_videos(
    (Video_path, "Original Video"),
    (Avg_bg_video_path, "Average Background"),
    (change_video_path, "Change Detection")
)

## Threshold Analysis

The change detection process is repeated using different threshold values, ranging from 5 to 50, in order to evaluate their impact on the results.

Lower thresholds (e.g., 5) produce a large amount of noise, visible as many white flickering pixels, since even small intensity variations are detected as motion.  
On the other hand, higher thresholds (e.g., 50) are too restrictive and fail to capture all relevant motion in the scene.

By comparing these results, a threshold value of 25 was selected for the video_change_avg, as it provides the best balance between noise reduction and accurate motion detection.

All the generated videos for the different thresholds are saved in the folder:
`video_output/threshold/`



In [ ]:
thresholds = [5, 10, 20, 30, 50]

# Create output directory for threshold videos
threshold_output_dir = "video/output/threshold"
os.makedirs(threshold_output_dir, exist_ok=True)

for t in thresholds:
    video = cv2.VideoCapture(Video_path)
    bg_video = cv2.VideoCapture(Avg_bg_video_path) 

    out_path = f"{threshold_output_dir}/threshold_{t}.mp4"
    out = cv2.VideoWriter(out_path, fourcc, fps, (w, h), isColor=False)

    while True:
        ret, frame = video.read()
        _, bg_frame = bg_video.read()
        if not ret:
            break    
        frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        bg_gray = cv2.cvtColor(bg_frame, cv2.COLOR_BGR2GRAY)
        
        diff = cv2.absdiff(frame_gray, bg_gray)
        _, mask = cv2.threshold(diff, t, 255, cv2.THRESH_BINARY)
        out.write(mask)

    video.release()
    bg_video.release()
    out.release()

# Show all threshold videos
show_videos(*[(f"{threshold_output_dir}/threshold_{t}.mp4", f"Threshold = {t}") for t in thresholds])

## Map Cleaning with Morphological Operations

In this step, morphological operations are applied to the binary change detection map in order to reduce noise and improve blob quality.

Erosion is first applied to remove small noisy regions by shrinking the detected blobs.  
Then, dilation is applied to expand the remaining blobs, helping to reconnect nearby regions that likely belong to the same object.

This combination allows for cleaner and more compact blob representations, improving the quality of subsequent processing steps.

The resulting video is saved as **"change_cleaned"**.

In [ ]:
cleaned_video_path = "video/output/change_cleaned.mp4"
out_cln = cv2.VideoWriter(cleaned_video_path, fourcc, fps, (w, h), isColor=False)

change = cv2.VideoCapture(change_video_path)

# Create kernel for morphological operations
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))

while True:
    ret, frame = change.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    eroded = cv2.erode(gray, kernel, iterations=1)
    dilated = cv2.dilate(eroded, kernel, iterations=2)

    # Mask = original expanded slightly → allows nearby blobs to connect
    # but still prevents runaway expansion far from any blob
    mask = cv2.dilate(gray, kernel, iterations=2)
    clean = cv2.bitwise_and(dilated, mask)

    out_cln.write(clean)

# Release resources
change.release()
out_cln.release()

show_videos(
    (Video_path, "Original Video"),
    (change_video_path, "Original Change Detection"),
    (cleaned_video_path, "Cleaned Change Detection")
)

In [ ]:
# Open the cleaned video
video = cv2.VideoCapture("video/change_cleaned.mp4")

# Get video properties
fps = int(video.get(cv2.CAP_PROP_FPS))
w = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Create output video with colored blobs for visualization
out_blobs = cv2.VideoWriter("video/change_blobs_visualized.mp4", 
                            cv2.VideoWriter_fourcc(*"avc1"), 
                            fps, (w, h))

frame_count = 0

while True:
    ret, frame = video.read()
    if not ret:
        break
    
    frame_count += 1
    
    # Convert to grayscale if needed
    if len(frame.shape) == 3:
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    else:
        gray = frame.copy()
    
    # ── COMPUTE CONNECTED COMPONENTS (BLOBS) ────────────────────────
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
        gray, connectivity=8
    )
    
    # ── VISUALIZATION ────────────────────────────────────────────────
    # Create colored visualization of blobs
    vis_frame = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    
    # Draw bounding boxes and labels for each blob (skip label 0 = background)
    for label in range(1, num_labels):
        x = stats[label, cv2.CC_STAT_LEFT]
        y = stats[label, cv2.CC_STAT_TOP]
        width = stats[label, cv2.CC_STAT_WIDTH]
        height = stats[label, cv2.CC_STAT_HEIGHT]
        area = stats[label, cv2.CC_STAT_AREA]
        cx, cy = centroids[label]
        
        # Draw bounding box (green)
        cv2.rectangle(vis_frame, (x, y), (x + width, y + height), (0, 255, 0), 2)
        
        # Draw centroid (red)
        cv2.circle(vis_frame, (int(cx), int(cy)), 3, (0, 0, 255), -1)
        
        # Add label with area
        cv2.putText(vis_frame, f"A:{area}", 
                   (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 0), 1)
    
    # Add frame info
    cv2.putText(vis_frame, f"Frame: {frame_count} | Blobs: {num_labels - 1}", 
               (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
    
    out_blobs.write(vis_frame)

# Release resources
video.release()
out_blobs.release()

# ── SHOW VISUALIZATION ──────────────────────────────────────────────
show_media(
    ("video/change_cleaned.mp4", "Cleaned Change Detection"),
    ("video/change_blobs_visualized.mp4", "Connected Components (Blobs)")
)

NameError: name 'show_media' is not defined